# FreshQA Experiment Runner

This notebook allows you to run experiments using different models and prompt types. You can select a judge model to evaluate the results and generate a report based on the experiments conducted.

In [28]:
# Colab / environment setup
import os, sys

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False
print(f"Running in Colab: {IN_COLAB}")

# 1. Ensure repository is cloned when running in Colab
if IN_COLAB:
    # REPLACE with your actual GitHub repository URL
    repo_url = "https://github.com/RephaelBlank/Agentic_Search.git"
    repo_name = "Agentic_Search" # Changed from "agentic_search" to "Agentic_Search"
    repo_dir = f"/content/{repo_name}"

    # Clone or pull latest changes
    if not os.path.exists(repo_dir):
        print(f"Cloning the repository {repo_name}...")
        os.system(f"git clone {repo_url}")
    else:
        print("Repository already cloned. Pulling latest changes...")
        os.system(f"cd {repo_dir} && git pull")

    # CRITICAL: Change current working directory to the cloned repo.
    # This makes your local path logic below work perfectly in Colab.
    os.chdir(repo_dir)

# Optional: mount Google Drive if your data is in Drive
# from google.colab import drive
# drive.mount('/content/drive')

# 2. Dynamic path computation
# Now 'cwd' is either your local directory or the cloned repo in Colab
cwd = os.getcwd()
notebooks_dir = os.path.join(cwd, 'new_order', 'colab-freshqa-experiments', 'notebooks')

# Try to compute repo root relative to notebooks location; fall back to cwd
if os.path.isdir(notebooks_dir):
    repo_root = os.path.abspath(os.path.join(notebooks_dir, '..'))
else:
    repo_root = cwd

src_path = os.path.abspath(os.path.join(repo_root, 'src'))
project_path = os.path.abspath(os.path.join(repo_root))

print('Computed project_path:', project_path)
print('Computed src_path:', src_path)

# 3. Add paths to sys.path so modules can be imported
if src_path not in sys.path:
    sys.path.insert(0, src_path)
    sys.path.insert(0, project_path)

# Optional: install requirements dynamically based on computed path
# req_path = os.path.join(project_path, 'requirements.txt')
# if IN_COLAB and os.path.exists(req_path):
#     print("Installing requirements...")
#     os.system(f"pip install -r {req_path} -q")

# 4. Safely import configuration now that paths are established
import src.config
print("Environment setup complete. Configuration loaded.")

Running in Colab: True
Repository already cloned. Pulling latest changes...
Computed project_path: /content/Agentic_Search
Computed src_path: /content/Agentic_Search/src
Environment setup complete. Configuration loaded.


In [23]:
!pip install litellm

In [29]:
# Import necessary libraries and project functions
import os
import json
import pandas as pd
from src.runner import run
from src.judge import run_judge


In [25]:
from google.colab import drive
drive.mount('/content/drive')

# Load the filtered questions (with a small fallback check)
data_path = '/content/drive/MyDrive/ANLP/Data/freshqa_filtered.csv'

questions = pd.read_csv(data_path)
print(f'Loaded {len(questions)} questions from {data_path}.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded 182 questions from /content/drive/MyDrive/ANLP/Data/freshqa_filtered.csv.


In [26]:
# Select model, prompt type, and judge model
model_options = ['llama-3-8b']  # Add more models as needed
prompt_variants = ['as_of']  # Add more prompt types as needed
judge_model = 'gemini/gemini-2.5-flash'  # Select judge model

print('Available models:', model_options)
print('Available prompt variants:', prompt_variants)
print('Selected judge model:', judge_model)

Available models: ['llama-3-8b']
Available prompt variants: ['as_of']
Selected judge model: gemini/gemini-2.5-flash


In [31]:
import os
os.system("git pull")
import importlib
import src.runner
importlib.reload(src.runner)
from src.runner import run

In [32]:
# Run experiments (this uses the lightweight runner that ships in src/)
n_questions = 5  # Number of questions to sample
seed = 42  # Random seed for reproducibility
results_path = 'results.json'  # Path where runner will save results
output_path = 'judgements.json'  # Path to save judgements

for model in model_options:
    for prompt in prompt_variants:
        print(f'Running experiments for model: {model}, prompt: {prompt}')
        run(n=n_questions, seed=seed, model=model, prompt=prompt, results_path=results_path, data_path=data_path)

# Run judging on results
if os.path.exists(results_path):
    run_judge(results_path=results_path, output_path=output_path)
    print('Judgements saved to', output_path)
else:
    print('No results file found at', results_path, '— skipping judging step.')

Running experiments for model: llama-3-8b, prompt: as_of
Saved 5 results to results.json
Saved 5 judgements to judgements.json
Judgements saved to judgements.json


## Conclusion
After running the experiments, the results will be evaluated, and judgements will be saved in the specified output file.